[Guide Index](README.md)

# Appendix D: How StarLayer classifies errors

When something goes wrong, StarLayer raises an error. This appendix explains the error classes, how to tell which one applies, and how to catch them.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01.01-getting-started.ipynb). Run cells from top to bottom, starting with the setup cell below.

The setup cell defines `show()`, which runs a call that is expected to fail and prints the error's class, its `kind`, its `context` and its message, and `list_kinds()`, which prints the kinds of an error class with their meanings. Some cells call `show()` on small examples that are meant to fail.

In [1]:
# Setup: run this cell first. It imports the error classes and defines two helpers used below:
#   show(label, call)       runs a call that is expected to fail and prints the error's class, kind, context and message
#   list_kinds(registry)    prints the kinds of an error class with their meanings
import textwrap

from starlayer.errors import ExecutionError, FormatError, ShapeError, StarLayerError
from starlayer.graph import XSD, StarLayerGraph, literal, parse_term, to_python
from starlayer.shacl import meta_validate
from starlayer.sparql import sparql_to_tree, srl, tree_to_sparql
from starlayer.starontology.manchester import manchester_to_tree


def list_kinds(registry):
    """Print the kinds of an error class with their meanings, read from its registry."""
    for kind, meaning in registry.items():
        print(textwrap.fill(meaning, width=100, initial_indent=f"{kind:18}", subsequent_indent=" " * 18))


def show(label, call):
    """Run a call that is expected to fail, and print what was raised."""
    try:
        call()
    except Exception as error:
        kind = getattr(error, "kind", None)
        position = f" line {error.line}" if getattr(error, "line", None) else ""
        context = getattr(error, "context", None)
        print(f"{label}")
        print(f"   {type(error).__name__}, kind={kind!r}, context={context!r}{position}")
        message = getattr(error, "why", None) or str(error).splitlines()[0]
        print(f"   {message[:100]}")
    else:
        print(label, "-> no error")
    print()


g = StarLayerGraph()
print ("example graph")
g.parse(data="@prefix ex: <http://example.org/> . ex:a ex:p ex:b .", format="turtle")
print (g.serialize())

example graph
@prefix ex: <http://example.org/> .
ex:a ex:p ex:b .



## D.1 Error types

A StarLayer call can raise two groups of errors.

- **A wrong argument** is a `TypeError` or a `ValueError`, the errors Python itself uses. The function refuses what it was given, and changing the argument resolves the error. D.2 explains the difference between a wrong type and a wrong value.

- **Anything else StarLayer raises on purpose** derives from `StarLayerError`, so `except StarLayerError` catches all of it. D.3 describes its subclasses.

Other Python errors can also arise, such as an `ImportError` when an optional package is missing or a `FileNotFoundError` for a file that does not exist. All of these classes derive from Python's `Exception`, so `except Exception` catches everything. `except StarLayerError` catches only what StarLayer raises on purpose. The cell below catches in layers, from the most specific to the most general.

In [2]:
def attempt(label, call):
    """Run a call and say which group of errors it raised."""
    try:
        call()
    except StarLayerError as error:  # StarLayer's own errors
        kind = getattr(error, "kind", None)
        print(f"{label}: {type(error).__name__}" + (f", kind={kind!r}" if kind else ""))
    except (TypeError, ValueError) as error:  # a wrong argument
        print(f"{label}: wrong argument, {type(error).__name__}")
    except Exception as error:  # any other Python error
        print(f"{label}: other error, {type(error).__name__}")
    else:
        print(label, "-> no error")


attempt("malformed SPARQL", lambda: sparql_to_tree("this is not sparql"))
attempt("a number instead of text", lambda: parse_term(5))
attempt("a file that does not exist", lambda: StarLayerGraph().parse("no-such-file.ttl", format="turtle"))
attempt("a valid query", lambda: sparql_to_tree("SELECT * WHERE { ?s ?p ?o }"))

malformed SPARQL: FormatError, kind='sparql'
a number instead of text: wrong argument, TypeError
a file that does not exist: other error, FileNotFoundError
a valid query -> no error


## D.2 Type and value

A wrong argument is a `TypeError` or a `ValueError`. The function refuses what it was given, and changing the argument may resolve the error. These are not `StarLayerError`s, because the cause is in the call and not in the data. The two differ in what is wrong:

- The *type* is the kind of thing that was passed: text (`str`), a number (`int`), a graph. A `TypeError` means the wrong kind of thing was passed, such as a graph object passed where text is expected.
- The *value* is the content of something of the right type. A `ValueError` means the type is right but the value is one the call does not accept, such as a negative number where zero or more is expected, or the text `"up"` for a base direction that must be `ltr` or `rtl`. Options that do not go together are also a `ValueError`.

Text in a defined format whose content cannot be read is neither: it is a `FormatError`, described in D.3.2.

Below are examples of `ValueError` and `TypeError`. The last example shows a `FormatError`, a string that cannot be parsed, for contrast.

In [3]:
show("a literal with a language and a datatype", lambda: literal("5", XSD.integer, lang="sv"))
show("a base direction that is not ltr or rtl", lambda: literal("x", lang="sv", direction="up"))
show("parse_term given a number", lambda: parse_term(5))
show("the same direction inside text that is parsed", lambda: parse_term('"x"@sv--up'))

a literal with a language and a datatype
   ValueError, kind=None, context=None
   a literal has a language tag or a datatype, not both: lang='sv' with datatype <http://www.w3.org/200

a base direction that is not ltr or rtl
   ValueError, kind=None, context=None
   direction must be 'ltr' or 'rtl', not 'up'

parse_term given a number
   TypeError, kind=None, context=None
   parse_term() needs a str, not int

the same direction inside text that is parsed
   FormatError, kind='rdf', context='term text'
   not exactly one RDF term in Turtle syntax



## D.3 The StarLayer errors

### D.3.1 `StarLayerError`

Everything StarLayer raises on purpose derives from `StarLayerError`, so `except StarLayerError` catches all of it. The class says what kind of problem it is:

- `FormatError`: text or data is malformed, such as a string that cannot be parsed as SPARQL or Turtle.
- `ShapeError`: the shapes graph is wrong, such as shapes that do not conform to the SHACL-SHACL shapes.
- `ExecutionError`: the arguments are correct but the work could not be done, such as an `owl:imports` that cannot be loaded.
- `EndpointError`: a SPARQL endpoint failed, such as an HTTP error answer, a host that cannot be reached or an update sent to an endpoint that has no update URL.

Each carries a `kind`, a short name for the cause, and a `context`, where it happened. The kinds, with what each one means, are held in a registry. (see below)

### D.3.2 `FormatError`: the text or data is malformed

A `FormatError` is raised when StarLayer is not able to parse or serialize with a specific format. 

`FormatError` has a `kind` for the language or format that is malformed. `FormatError.KINDS` maps each kind to its meaning. The list below is read from the registry.

Context carries `line` and `col` when the parser knows where the problem is, and `context` says which input it was (for example "language tag" or "Turtle document").



In [4]:
list_kinds(FormatError.KINDS)

sparql            Malformed SPARQL text, sqe: tree or VERSION declaration
srl               Malformed SRL text, srl: tree, DATA block or rule body element, or a rule set that
                  breaks the SRL rules
rdf               Malformed RDF document, term text (IRI, language tag, lexical form), list or
                  prefixed name
manchester        Malformed Manchester Syntax text, manch: tree or SWRL Rule: frame
owl               Invalid OWL in RDF, or a malformed owl:imports declaration


Below, one malformed input of each of the most common kinds is read, and the class, kind, context and line are printed.

In [5]:
show("SPARQL text that is not SPARQL", lambda: sparql_to_tree("this is not sparql"))
show("a query that contradicts its VERSION line", lambda: g.query('VERSION "1.1" SELECT ?s WHERE { ?s ?p <<( ?s ?p ?o )>> }'))
show("a Turtle document with a syntax error", lambda: StarLayerGraph().parse(data="<a> <b>", format="turtle"))
show("text that is not exactly one term", lambda: parse_term("4 5"))
show("a lexical form that does not fit its datatype", lambda: to_python(parse_term('"abc"^^xsd:integer', {"xsd": str(XSD)})))
show("an unfinished SRL rule", lambda: srl.parse_ruleset("RULE { ?x"))
show("Manchester text with a syntax error", lambda: manchester_to_tree("Prefix: : <http://example.org/>\nClass: A SubClassOf:"))
show("a SWRL rule in Manchester text (not supported)", lambda: manchester_to_tree("Prefix: : <http://example.org/>\nRule: Person(?x)"))
show("a graph that holds no tree to write", lambda: tree_to_sparql(StarLayerGraph()))

SPARQL text that is not SPARQL
   FormatError, kind='sparql', context=None line 1
   expected SELECT, ASK, CONSTRUCT or DESCRIBE (found 'this')

a query that contradicts its VERSION line
   FormatError, kind='sparql', context='VERSION'
   SPARQL query declares VERSION '1.1' but uses a triple term, which '1.1' conformance excludes (SPARQL

a Turtle document with a syntax error
   FormatError, kind='rdf', context='Turtle document' line 1
   Parser error at line 1 between columns 1 and 4: No scheme found in an absolute IRI (line 1)

text that is not exactly one term
   FormatError, kind='rdf', context='term text'
   not exactly one RDF term in Turtle syntax

a lexical form that does not fit its datatype
   FormatError, kind='rdf', context='xsd:integer literal'
   'abc' is not a valid integer

an unfinished SRL rule
   FormatError, kind='srl', context=None line 1
   expected a predicate (found end of input)

Manchester text with a syntax error
   FormatError, kind='manchester', context=Non

### D.3.3 `ExecutionError`: the work could not be done

The input is valid, but running it failed or reached a limit. The `kind` says why. `ExecutionError.KINDS` maps each kind to its meaning. 

In [6]:
list_kinds(ExecutionError.KINDS)

rule              Failed rule, or a rule set that fails as a whole (for example, one that cannot be
                  ordered)
expression        Node expression that cannot be evaluated
recursion         Shape that reaches itself through sh:not (a cycle in the data)
constraint-query  sh:sparql constraint query that reported ?failure true
internal          Bug in StarLayer (it asked itself for something it cannot do); report it
                  (report_url)
timeout           OWL-DL reasoning that exceeded its time budget
inconsistent      Data found logically inconsistent by the OWL-DL reasoner
inference         Reasoner that could not run or died (Java missing or broken, or the subprocess
                  exited without a result)
query             SPARQL engine failure on a query or update whose text parsed
import            Unresolved owl:imports or SRL IMPORTS
unsupported       Valid input that the library or the target notation cannot handle
limit             Size or complexity ca

Two of the kinds are easy to confuse:
- **`unsupported`**: *the user* asked for something StarLayer cannot do. The input is valid, and the user can see the limit and choose differently: a regime that is not registered, a format that cannot hold the data, a Manchester construct with no counterpart.
- **`internal`**: *StarLayer* asked itself for something it cannot do. It is a bug in StarLayer, the user cannot fix it, and the message says where to report it (`error.report_url` holds the address, so an application can show it).

The `context` says where it happened, such as the rule, the shape or the call.

An `import` error also carries `unresolved`, the tuple of import IRIs that could not be resolved, so an application can load them and call again.

Below, one example each of `query`, `limit`, `rule`, `import` and `unsupported` (twice).

In [7]:
show("a custom function that was not passed", lambda: list(g.query("SELECT (<urn:fn:x>(?s) AS ?y) WHERE { ?s ?p ?o }")))
show("a query nested far too deeply (kind limit)", lambda: sparql_to_tree("SELECT ?x WHERE " + "{ " * 3000 + "?x ?p ?o" + " }" * 3000))
show("a rule set that cannot be ordered", lambda: srl.parse_ruleset("PREFIX ex: <http://example.org/> RULE { ?x ex:f ex:y } WHERE { ?x a ex:C . NOT { ?x ex:f ex:y } }").plan())
show("a rule set with imports that were never resolved", lambda: srl.parse_ruleset("IMPORTS <http://example.org/rules> RULE { ?x <urn:p> ?y } WHERE { ?x <urn:q> ?y }").infer(g))

labelled = StarLayerGraph()
labelled.parse(data='@prefix : <http://x/> . @prefix owl: <http://www.w3.org/2002/07/owl#> . @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> . :A a owl:Class ; rdfs:label "x"@ar--rtl .', format="turtle")
show("an entailment name that is not a registered regime", lambda: g.infer(entailment="nonsense"))
show("a graph that Manchester Syntax cannot write", lambda: labelled.serialize(format="manchester"))

a custom function that was not passed
   ExecutionError, kind='query', context='Oxigraph query'
   The custom function <urn:fn:x> is not supported

a query nested far too deeply (kind limit)
   ExecutionError, kind='limit', context='SPARQL parse nesting depth'
   SPARQL expression or pattern nested too deeply to parse

a rule set that cannot be ordered
   ExecutionError, kind='rule', context='stratification'
   SRL: stratification condition violated - a recursive dependency involves a closed edge, so this rule

a rule set with imports that were never resolved
   ExecutionError, kind='import', context='IMPORTS <http://example.org/rules>'
   SRL: IMPORTS <http://example.org/rules> is not resolved: the rule set was not resolved before it ran

an entailment name that is not a registered regime
   ExecutionError, kind='unsupported', context='entailment regime'
   Unsupported entailment regime(s) 'nonsense'; the available regimes are ['http://www.w3.org/ns/entail

a graph that Manchester Syn

### D.3.4 `ShapeError`: the shapes are wrong

`ShapeError` is about a shapes graph. Its `kind` says where the problem was found. `ShapeError.KINDS` maps each kind to its meaning. 

In [ ]:
list_kinds(ShapeError.KINDS)

compile           Malformed or unsupported shapes: a bad shape, path, constraint parameter, query or
                  rule; an unsupported construct; an owl:imports object that is not an IRI
meta              Shapes graph that is invalid against the SHACL-SHACL meta shapes
                  (meta_shacl=True); problems and the report say how


A `compile` error is raised when the SHACL engine cannot read the shapes graph. The engine cannot process it, so it cannot produce a validation report: the error carries a message and a `context`. A `meta` error also carries a validation report of the problems it found; see [05.02.4](05.02-starlayer-shacl-engine.ipynb).

The `context` says where in the shapes graph it was found, such as a shape or a rule.

Below, a node shape with a `sh:path` raises a `compile` error when the processor is created, and a shapes graph with an invalid `sh:datatype` raises a `meta` error.

In [ ]:
from starlayer.shacl import StarShaclProcessor

misplaced = StarLayerGraph()
misplaced.parse(data="@prefix ex: <http://example.org/> . @prefix sh: <http://www.w3.org/ns/shacl#> . ex:S a sh:NodeShape ; sh:path ex:p .", format="turtle")
try:
    StarShaclProcessor(shacl_graph=misplaced)
except ShapeError as error:
    print(type(error).__name__, error.kind)
    print("message:", error)

ShapeError compile
message: http://example.org/S: a sh:NodeShape cannot be the subject of sh:path


In [8]:
bad_shapes = StarLayerGraph()
bad_shapes.parse(data="@prefix sh: <http://www.w3.org/ns/shacl#> . @prefix ex: <http://example.org/> . ex:S a sh:NodeShape ; sh:property [ sh:path ex:p ; sh:datatype 5 ] .", format="turtle")
try:
    meta_validate(bad_shapes)
except ShapeError as error:
    print(type(error).__name__, error.kind)
    print("problems:", error.problems)

ShapeError meta
problems: ('the property shape of ex:S for ex:p, sh:datatype: violates sh:or',)


### D.3.5 `EndpointError`: a SPARQL endpoint failed

`EndpointError` is raised by a call to a SPARQL endpoint. Its `kind` says how the call failed. `EndpointError.KINDS` maps each kind to its meaning. The list below is read from the registry, so it is always the list the library uses.

In [ ]:
from starlayer.errors import EndpointError

list_kinds(EndpointError.KINDS)

http              Error status answered by the endpoint (status holds it)
unreachable       Host that cannot be reached: the connection failed or timed out
content-type      Answer in a content type StarLayer cannot read
read-only         Update sent to an endpoint that has no update URL (a read-only endpoint)


`EndpointError` is also an `OSError`, because a network failure is an I/O error, so code that catches `OSError` for network failures keeps working. For the kind `http`, `status` holds the HTTP status code. The `context` is the URL of the endpoint.

An update sent to an endpoint that has no update URL is kind `read-only`: StarLayer refuses it before sending anything, so there is no `status`. A read-only server that refuses an update is kind `http`, with its status (403 or 405).

## D.4 Catching errors

Catch the narrowest class that can be handled. `except StarLayerError` is the catch-all for the library. When a handler is for only one `kind`, call `reraise_unless(error, "kind")` first (it is imported from `starlayer.errors`): it returns if the error has the kind given and raises the error again if its kind is a different one, so a handler never swallows a cause it was not written for.

Below, each text in a list is read as a tree and then run as a query. Each error passes through a series of nested handlers, from the narrowest to the widest. A handler that catches the error prints its name and the kind of the error, and passes the error on, so the output lists every level that would catch it: first the class (`FormatError` or `ExecutionError`), then `StarLayerError`, then `Exception`. A blank line separates the texts, and each text is printed only as far as its first 50 characters.

In [ ]:
from contextlib import contextmanager


@contextmanager
def level(name, classes):
    """One handler: report an error of these classes with its kind, if it has one, then pass it on."""
    try:
        yield
    except classes as error:
        kind = getattr(error, "kind", None)
        print(f"   caught by {name}" + (f", kind {kind!r}" if kind else ""))
        raise


def read(text):
    """Read the text as a tree, then run it as a query."""
    sparql_to_tree(text)
    return list(g.query(text))


QUERIES = [
    "SELECT ?s WHERE { ?s ?p ?o }",
    "not a query",
    'VERSION "1.1" SELECT ?s WHERE { ?s ?p <<( ?s ?p ?o )>> }',
    "SELECT (<urn:fn:x>(?s) AS ?y) WHERE { ?s ?p ?o }",
    "SELECT ?x WHERE " + "{ " * 3000 + "?x ?p ?o" + " }" * 3000,
]

for text in QUERIES:
    print(text[:50])
    try:
        # the handlers are nested: the error reaches the innermost first and goes outward
        with (
            level("except Exception", Exception),
            level("except StarLayerError", StarLayerError),
            level("except FormatError", FormatError),
            level("except ExecutionError", ExecutionError),
        ):
            read(text)
    except Exception:
        pass
    else:
        print("   no error")
    print()

SELECT ?s WHERE { ?s ?p ?o }
   no error

not a query
   caught by except FormatError, kind 'sparql'
   caught by except StarLayerError, kind 'sparql'
   caught by except Exception, kind 'sparql'

VERSION "1.1" SELECT ?s WHERE { ?s ?p <<( ?s ?p ?o
   caught by except FormatError, kind 'sparql'
   caught by except StarLayerError, kind 'sparql'
   caught by except Exception, kind 'sparql'

SELECT (<urn:fn:x>(?s) AS ?y) WHERE { ?s ?p ?o }
   caught by except ExecutionError, kind 'query'
   caught by except StarLayerError, kind 'query'
   caught by except Exception, kind 'query'

SELECT ?x WHERE { { { { { { { { { { { { { { { { { 
   caught by except ExecutionError, kind 'limit'
   caught by except StarLayerError, kind 'limit'
   caught by except Exception, kind 'limit'

